In [3]:
pip install arch pyvinecopulib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 982.9/982.9 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 33.3 MB/s eta 0:00:00


In [4]:
import numpy as np
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
from arch import arch_model
import pyvinecopulib as pv
from scipy.stats import genpareto, kendalltau
import os
from dataclasses import dataclass, field, asdict
from typing import Optional, Sequence, Union, List, Dict, Any, Tuple
import numpy as np
import cvxpy as cp
from scipy.spatial.distance import squareform
from scipy.cluster.hierarchy import linkage, fcluster, leaves_list
from sklearn.covariance import LedoitWolf

In [8]:
@dataclass
class HERCParams:
  quantile:float|None=0.95
  use_lw_shrinkage:bool=False
  k_max:int=5
  n_sims:int=100

In [6]:
@dataclass
class ClusterParams:
  quantiles: dict[str, float] = field(default_factory=dict)
  paths: np.ndarray = field(default_factory=np.ndarray)


In [11]:
class ClusterEngine:
  def __init__(self, params: HERCParams, debug=False, **kwargs):
    super().__init__(params=params, debug=debug, **kwargs)
    self.cp = ClusterParams()
    self.p = params

  def _get_tail_mask(self, sim_returns, assets, w=None):
    if w is None:
      w = np.ones(len(assets)) / len(assets)

    asset_R = np.prod(1 + sim_returns, axis=2) - 1
    alpha = self.p.alpha_threshold * 100

    p_R = asset_R @ w

    var_p = np.percentile(p_R, alpha)
    tail_mask = p_R >= var_p

    return tail_mask

  def _get_dist_mtx(self, sim_returns, assets, w=None):
    tail_mask = self._get_tail_mask(sim_returns, assets, w)

    tau_mtx = np.zeros((len(assets), len(assets)))
    tail_returns = sim_returns[tail_mask, :]
    asset_losses = -tail_returns

    for i in range(len(assets)):
      for j in range(len(assets)):
        tau_ij, _ = kendalltau(
          asset_losses[:, i],
          asset_losses[:, j]
        )

        tau_mtx[i, j] = tau_ij

    dist_mtx = np.sqrt(0.5 * (1 - tau_mtx))

    return dist_mtx